get the data


In [1]:
import os
os.chdir('/home/cyf/wbi/Virginia/code/wbi_0123/wholistic_registration/src/wholistic_registration')
from utils import IO
from utils import calFlowCrossResolution, calFlow3d_Wei_v1
from utils import registration
from utils import option
from utils import preprocess as prep
from utils import mask
from utils import  visualization
import numpy as np
gut_mov_Path     = "/home/cyf/wbi/Virginia/registrated_data/f260201/gut/raw/260201_test1_0_00002_TZCXY.ome.tif"
ventral_mov_Path = "/home/cyf/wbi/Virginia/registrated_data/f260201/ventral/raw/260201_test1_0_00003_TZCXY.ome.tif"
dorsal_mov_Path  = "/home/cyf/wbi/Virginia/registrated_data/f260201/dorsal/raw/260201_test1_0_00005_TZCXY.ome.tif"

gut_ref_Path     = "/home/cyf/wbi/Virginia/registrated_data/f260201/gut/anat/260201_test1_0_00002_TZCXY.ome.tif"
ventral_ref_Path = "/home/cyf/wbi/Virginia/registrated_data/f260201/ventral/anat/260201_test1_0_00002_TZCXY.ome.tif"
dorsal_ref_Path  = "/home/cyf/wbi/Virginia/registrated_data/f260201/dorsal/anat/260201_test1_0_00004_TZCXY.ome.tif"


CuPy is available with CUDA - using GPU acceleration


gut exp

In [15]:
gut_ref,gut_ref_desc = IO.readTifff(gut_ref_Path)
gut_mov,gut_mov_desc = IO.readTifff(gut_mov_Path)

gut_ref = gut_ref.transpose(1,2,0)
gut_mov = gut_mov.transpose(0,2,3,1)
### see the slice 86, 87,88
### use the whole image as the groundtruth
#initial the pyramid parameters
for frame in range(0,5):
    gut_ref_sample = gut_ref[...,40:120]
    gut_mov_sample = gut_mov[frame,:,:,40:120]
    print(gut_ref_sample.shape)
    print(gut_mov_sample.shape)
    from skimage.exposure import match_histograms
    gut_mov_sample = match_histograms(gut_mov_sample, gut_ref_sample)
    # visualization.visualize_2d_image(gut_mov_sample[...,slice],title = "moving image")
    # visualization.visualize_2d_image(gut_ref_sample[...,slice],title = "reference image")
    slice = 39
    option['motion']=np.zeros([gut_ref_sample.shape[0],gut_ref_sample.shape[1],gut_ref_sample.shape[2],3])
    option['r']=5
    option['layer']=2
    option['iter']=10
    option['movRange']=5.
    option['tol']=1e-6
    thresFactor= 5.
    maskRange  = [5.,4000.]
    smoothPenalty_raw=0.03

    option['mask_ref']=mask.getMask(gut_ref_sample,thresFactor)
    option['mask_ref']=mask.bwareafilt3_wei(option['mask_ref'],maskRange)
    option['mask_mov']=mask.getMask(gut_mov_sample,thresFactor)
    option['mask_mov']=mask.bwareafilt3_wei(option['mask_mov'],maskRange)

    print(option['mask_ref'].shape)
    Pnltfactor = prep.getSmPnltNormFctr(gut_ref_sample, option)
    smoothPenalty=Pnltfactor*smoothPenalty_raw
    option['smoothPenalty']=smoothPenalty
    option['zRatio'] = 1

    motion_current, _ , new_coords,error_logs = calFlow3d_Wei_v1.getMotion(gut_mov_sample, gut_ref_sample,option,verbose = True)
    corrected_mem = calFlow3d_Wei_v1.correctMotion(gut_mov_sample, motion_current)
    # visualization.visualize_2d_image(corrected_mem[...,47],title = "corrected image")

    gut_single_plane = gut_mov_sample[:,:,slice:slice+1]
    option['zRatio_HR']=1

    H, W, D = gut_single_plane.shape
    X, Y, Z = np.indices((H, W, D))

    phase_init = np.stack([
        X.astype(np.float32),
        Y.astype(np.float32),
        (slice-1 + Z * option["zRatio"] / option["zRatio_HR"]).astype(np.float32)
    ], axis=-1)
    print(phase_init.shape)
    option["phase"] = np.array(phase_init)

    phase_new,motion_current,data_mov_mapped= calFlowCrossResolution.getMotion(gut_single_plane, gut_ref_sample,option,verbose = True)
    # visualization.visualize_2d_image(data_mov_mapped.get(),title = "mapped image")
    IO.write_multichannel_volume_as_ome_tiff(
            volume=[data_mov_mapped.get().squeeze(),gut_mov_sample[...,slice],gut_ref_sample[...,slice],corrected_mem[...,slice]],      # single channel
            out_dir="/home/cyf/wbi/Virginia/HR_exp/",
            frame_idx= frame,
            label=f'HR_S{slice}'
    )

(1086, 538, 80)
(1086, 538, 80)
(1086, 538, 80)
starting layer 2 out of 2
Downsample layer: 2	Iter: 0	Error: 261700.665, Diff Error: 261700.665, Penalty Error: 0.000
Downsample layer: 2	Iter: 0	Max motion: 7.17	Max diff. old vs new motion: 7.1711
Downsample layer: 2	Iter: 1	Error: 160007.854, Diff Error: 159417.258, Penalty Error: 590.596
Downsample layer: 2	Iter: 1	Max motion: 10.37	Max diff. old vs new motion: 8.9982
Downsample layer: 2	Iter: 2	Error: 108912.894, Diff Error: 107171.116, Penalty Error: 1741.777
Downsample layer: 2	Iter: 2	Max motion: 14.69	Max diff. old vs new motion: 6.4726
Downsample layer: 2	Iter: 3	Error: 81789.058, Diff Error: 79083.237, Penalty Error: 2705.821
Downsample layer: 2	Iter: 3	Max motion: 18.41	Max diff. old vs new motion: 5.3194
Downsample layer: 2	Iter: 4	Error: 67672.355, Diff Error: 64240.385, Penalty Error: 3431.969
Downsample layer: 2	Iter: 4	Max motion: 22.01	Max diff. old vs new motion: 4.5229
Downsample layer: 2	Iter: 5	Error: 60792.138, Diff

In [ ]:
gut_ref,gut_ref_desc = IO.readTifff(gut_ref_Path)
gut_mov,gut_mov_desc = IO.readTifff(gut_mov_Path)

gut_ref = gut_ref.transpose(1,2,0)
gut_mov = gut_mov.transpose(0,2,3,1)
### see the slice 86, 87,88
### use the whole image as the groundtruth
#initial the pyramid parameters
for frame in range(1,5):
    gut_ref_sample = gut_mov[0,:,:,40:120]
    gut_mov_sample = gut_mov[frame,:,:,40:120]
    print(gut_ref_sample.shape)
    print(gut_mov_sample.shape)
    from skimage.exposure import match_histograms
    gut_mov_sample = match_histograms(gut_mov_sample, gut_ref_sample)
    # visualization.visualize_2d_image(gut_mov_sample[...,slice],title = "moving image")
    # visualization.visualize_2d_image(gut_ref_sample[...,slice],title = "reference image")
    slice = 39
    option['motion']=np.zeros([gut_ref_sample.shape[0],gut_ref_sample.shape[1],gut_ref_sample.shape[2],3])
    option['r']=5
    option['layer']=2
    option['iter']=15
    option['movRange']=10.
    option['tol']=1e-6
    thresFactor= 5.
    maskRange  = [5.,4000.]
    smoothPenalty_raw=0.1

    option['mask_ref']=mask.getMask(gut_ref_sample,thresFactor)
    option['mask_ref']=mask.bwareafilt3_wei(option['mask_ref'],maskRange)
    option['mask_mov']=mask.getMask(gut_mov_sample,thresFactor)
    option['mask_mov']=mask.bwareafilt3_wei(option['mask_mov'],maskRange)

    print(option['mask_ref'].shape)
    Pnltfactor = prep.getSmPnltNormFctr(gut_ref_sample, option)
    smoothPenalty=Pnltfactor*smoothPenalty_raw
    option['smoothPenalty']=smoothPenalty
    option['zRatio'] = 1

    motion_current, _ , new_coords,error_logs = calFlow3d_Wei_v1.getMotion(gut_mov_sample, gut_ref_sample,option,verbose = True)
    corrected_mem = calFlow3d_Wei_v1.correctMotion(gut_mov_sample, motion_current)
    # visualization.visualize_2d_image(corrected_mem[...,47],title = "corrected image")

    gut_single_plane = gut_mov_sample[:,:,slice:slice+1]
    option['zRatio_HR']=1

    H, W, D = gut_single_plane.shape
    X, Y, Z = np.indices((H, W, D))
    print(Z)
    phase_init = np.stack([
        X.astype(np.float32),
        Y.astype(np.float32),
        (slice + Z * option["zRatio"] / option["zRatio_HR"]).astype(np.float32)
    ], axis=-1)
    print(phase_init.shape)
    option["phase"] = np.array(phase_init)

    phase_new,motion_current,data_mov_mapped= calFlowCrossResolution.getMotion(gut_single_plane, gut_ref_sample,option,verbose = True)
    # visualization.visualize_2d_image(data_mov_mapped.get(),title = "mapped image")
    IO.write_multichannel_volume_as_ome_tiff(
            volume=[data_mov_mapped.get().squeeze(),gut_mov_sample[...,slice],gut_ref_sample[...,slice],corrected_mem[...,slice]],      # single channel
            out_dir="/home/cyf/wbi/Virginia/HR_exp/",
            frame_idx= frame,
            label=f'HR_S{slice}_align2First'
    )

(1086, 538, 80)
(1086, 538, 80)
(1086, 538, 80)
starting layer 1 out of 1
Downsample layer: 1	Iter: 0	Error: 16270.058, Diff Error: 16270.058, Penalty Error: 0.000
Downsample layer: 1	Iter: 0	Max motion: 3.99	Max diff. old vs new motion: 3.9946
Downsample layer: 1	Iter: 1	Error: 10088.615, Diff Error: 9946.039, Penalty Error: 142.576
Downsample layer: 1	Iter: 1	Max motion: 8.27	Max diff. old vs new motion: 5.7855
Downsample layer: 1	Iter: 2	Error: 8178.749, Diff Error: 7761.755, Penalty Error: 416.994
Downsample layer: 1	Iter: 2	Max motion: 10.06	Max diff. old vs new motion: 6.0848
Downsample layer: 1	Iter: 3	Error: 7438.062, Diff Error: 6807.574, Penalty Error: 630.488
Downsample layer: 1	Iter: 3	Max motion: 12.08	Max diff. old vs new motion: 5.4852
Downsample layer: 1	Iter: 4	Error: 7047.052, Diff Error: 6248.966, Penalty Error: 798.086
Downsample layer: 1	Iter: 4	Max motion: 14.56	Max diff. old vs new motion: 5.2362
Downsample layer: 1	Iter: 5	Error: 6800.455, Diff Error: 5873.662, 

KeyboardInterrupt: 